# 3-stage learned-router pipeline

This three-stage system trains at five noise levels and stores each condition under `output/three_stage/train_sigma...`. Each condition receives clean and noisy test evaluation under nested `test_sigma...` directories. The specialist decision is made by a learned router from Stage 1 predicted coefficients; true shape labels are used only as training labels for the router and for diagnostic per-shape reporting.

## Stage contracts

Stage 1 predicts coefficient features. A coefficient router predicts whether an input should use the two-circle specialist. The general head handles all inputs and the specialist head replaces the general prediction only when the learned router selects it. At inference, routing receives predicted coefficients only. Clean-test IoU is the official result; the five test-noise curves are supplementary robustness results.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import torch

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from config import Task9RunConfig, Task9StackConfig
from datasets import build_task9_general_dataset
from experiments import evaluate_noise_sweep, layer_table, parameter_count, save_noise_results, sigma_label
from experiments.notebook import plot_noise_curve, plot_shape_curves
from final_models.three_stage import run_three_stage_with_predictor
from models import Stage1Regressor
from models.three_models import CoefficientToGeneralMaskModel, CoefficientToSpecialistMaskModel

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTPUT = ROOT / 'output' / 'three_stage'
OUTPUT.mkdir(parents=True, exist_ok=True)
TRAIN_SIGMAS = (0.0, 0.001, 0.0025, 0.005, 0.01)
TEST_SIGMAS = TRAIN_SIGMAS
RUN_TRAINING = False

def make_config(train_sigma):
    return Task9RunConfig(N=10, training_samples=10_000, validation_samples=2_000, test_samples=1_000, noise_sigma=train_sigma, noise_mode='absolute', seed=42, use_validation_threshold_sweep=True, model=Task9StackConfig(routing_mode='predicted_router'), output_dir=OUTPUT / f'train_sigma{sigma_label(train_sigma)}')

stage1_check = Stage1Regressor(input_dim=22, output_dim=22, hidden_dims=(256, 512, 256), dropout_rates=(0.2, 0.2, 0.2))
general_check = CoefficientToGeneralMaskModel(input_dim=22, output_dim=32 * 32, hidden_dims=(512, 1024, 2048), dropout_rates=(0.2, 0.2, 0.2))
specialist_check = CoefficientToSpecialistMaskModel(input_dim=22, output_dim=32 * 32, hidden_dims=(1024, 2048, 4096), dropout_rates=(0.3, 0.3, 0.3))
for name, model, dummy in (('stage1', stage1_check, torch.zeros(2, 22)), ('general_head', general_check, torch.zeros(2, 22)), ('specialist_head', specialist_check, torch.zeros(2, 22))):
    print(name, parameter_count(model))
    layer_table(model, dummy).to_csv(OUTPUT / f'{name}_architecture.csv', index=False)

In [ ]:
comparison_rows = []
for train_sigma in TRAIN_SIGMAS:
    config = make_config(train_sigma)
    train_dir = config.output_dir
    summary_path = config.run_output_dir / 'summary.json'
    should_train = RUN_TRAINING or not summary_path.exists()
    print(f'\n=== training sigma={train_sigma:g} ===')
    if not should_train:
        print('Existing summary found; retraining this condition to recreate the live learned-router predictor.')
    summary, predict_logits = run_three_stage_with_predictor(config, device=DEVICE)
    bundle = build_task9_general_dataset(config)
    threshold = float(summary['threshold_summary']['selected_threshold'])
    overall, by_shape = evaluate_noise_sweep(predict_logits, bundle.test.gradient_data, bundle.test.masks, bundle.test.shape_types, threshold, noise_levels=TEST_SIGMAS, seed=42)
    save_noise_results(overall, by_shape, train_dir, stem='test_noise_robustness')
    plot_noise_curve(overall, train_dir / 'test_noise_curve.png', f'Three-stage test noise, train sigma={train_sigma:g}')
    plot_shape_curves(by_shape, train_dir / 'test_noise_by_shape.png', f'Three-stage per-shape noise, train sigma={train_sigma:g}')
    clean = overall[overall['noise_sigma'] == 0.0].iloc[0]
    comparison_rows.append({'train_sigma': train_sigma, 'clean_test_iou': clean['mean_iou'], 'clean_pixel_accuracy': clean['pixel_accuracy'], 'threshold': threshold})
comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(OUTPUT / 'training_noise_comparison.csv', index=False)
display(comparison)

The router is trained using Stage 1 predicted coefficients and the known training shape labels. Those labels are not passed to the router during validation or test inference. The five test levels are `0.0`, `0.001`, `0.0025`, `0.005`, and `0.01`.